# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and process the FAIR<sup>2</sup> Proteomics dataset ([Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)) using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema JSON-LD, compatible with `mlcroissant`. All datasets, fields, and columns processed here are referenced by their `@id` property for reproducibility and clarity.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review the available record sets and their fields. Each is referenced by its `@id`.

In [ ]:
# Examine available record sets and fields

def show_record_sets(dataset):
    print('Record Sets (`@id` and name):\n')
    for rs in dataset.record_sets:
        print(f"- {rs['@id']} | {rs['name']}")
        # Print a few fields for each record set
        if 'field' in rs:
            print("  Fields:")
            for field in rs['field']:
                if isinstance(field, dict):
                    print(f"    - {field.get('@id', '?')} | {field.get('name', '?')}")
                else:
                    print(f"    - {field}")
        print()

show_record_sets(dataset)

## 3. Data Extraction
Load data from each record set as a DataFrame for analysis. All `record_set` and field references use their Croissant `@id`.

In [ ]:
# List the record set IDs (from overview, use only top-level record sets you want to import, e.g. first/main)
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

# Load available record sets to DataFrames
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records for RecordSet '{record_set_id}' with columns: {list(dataframes[record_set_id].columns)}")
        else:
            print(f"No records found for '{record_set_id}'.")
    except Exception as e:
        print(f"Error loading RecordSet '{record_set_id}': {e}")

# Print columns of the first record set for demonstration
if record_set_ids and record_set_ids[0] in dataframes:
    first_rs_id = record_set_ids[0]
    print(f"\nColumns in '{first_rs_id}':\n{dataframes[first_rs_id].columns.tolist()}")
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply basic processing:

- Filter by a numeric field
- Normalize numeric values
- Optionally group by a category

In [ ]:
# Pick a numeric field for analysis (inspect output from Data Extraction)
# For demonstrative purposes, let's assume a field called 'Molecular_Weight' and group by 'Protein_Description'.
# Replace with the actual @id column names from your record set, e.g., 'cr:Molecular_Weight', 'cr:Protein_Description', etc.

record_set_id = first_rs_id
# Example field IDs (adjust as needed):
numeric_field_id = None
group_field_id = None

# Try to auto-detect a numeric field
df = dataframes[record_set_id]
potential_numeric = df.select_dtypes(include=['number']).columns
if len(potential_numeric) > 0:
    numeric_field_id = potential_numeric[0]
    print(f"Auto-selected numeric field: {numeric_field_id}")
else:
    print("No numeric fields detected.")
    numeric_field_id = df.columns[0]  # fallback

# Try to auto-detect a non-numeric/grouping field
potential_groups = df.select_dtypes(include=['object']).columns
if len(potential_groups) > 0:
    group_field_id = potential_groups[0]
    print(f"Auto-selected group field: {group_field_id}")
else:
    group_field_id = None

# Filtering and normalization
threshold = df[numeric_field_id].mean() if numeric_field_id in df.columns else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered {len(filtered_df)} records with '{numeric_field_id}' > {threshold:.2f}")

# Normalize the numeric field
filtered_df[numeric_field_id + "_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()

print(filtered_df[[numeric_field_id, numeric_field_id + "_normalized"]].head())

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    print(f"Grouped mean of '{numeric_field_id}' by '{group_field_id}':")
    print(grouped_df.head())

## 5. Visualization
Simple distribution and relationship plots using Matplotlib or Pandas plotting. All axes are labeled using the respective field `@id`.

In [ ]:
import matplotlib.pyplot as plt

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    df[numeric_field_id].hist(bins=40)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        df.boxplot(column=numeric_field_id, by=group_field_id, rot=90, figsize=(10,5))
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle("")
        plt.show()

## 6. Conclusion
- We loaded the FAIR<sup>2</sup> mass spectrometry dataset using its Croissant schema.
- The dataset provides protein-level attributes (such as molecular weight) suitable for exploration by `@id`.
- Initial EDA steps demonstrate filtering, normalization, and visualization, facilitating further biological or statistical analysis.

Continue to reference fields by their Croissant `@id` for your workflows. See [mlcroissant documentation](https://mlcommons.github.io/croissant/) for more utilities.